# Oakland corridor activity — raw pilot

Run this notebook from top to bottom. It tests **Fruitvale** for January 7–13, 2019 and January 8–14, 2024 using the established geohash9 corridor cover. The output is observed **raw corridor activity**, with one device × corridor × local calendar date counted as one visit. The five time periods (`all`, `weekdays`, `weekends`, `nine-five`, `evening`) overlap and cannot be summed.

The notebook writes monthly activity and half-year repeat-visitor summaries to `DEDICATED.EDDIT_EAST_BAY_TMP` and exports CSVs to `./activity/pilot_raw/`. The repeat measures reflect **only the sampled weeks**, so they test the method but do not estimate full half-year loyalty. No HOME table, HOME geography, or sample-normalization query is needed to run this pilot. The outputs are explicitly labelled raw.

The completed corridor-stop and device summaries from the previous pilot are reused by default after validating their dates and corridor. Set `REBUILD_PILOT_TABLES=True` to deliberately extract those two weeks again. This can take several minutes.

## 1. Connection and configuration

In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

%load_ext cuebiqmagic.magics
%init_cuebiq_data
snow_engine = get_ipython().user_ns["instance"]


schema : args.schema='paas_cda_pe_v3' and database: args.database='cuebiq_data' used
The connection to the Snowflake cluster has been established using database 'cuebiq_data' and schema 'paas_cda_pe_v3'.

You can now retrieve the connection using the following statement:
snow_engine = get_ipython().user_ns['instance']

The connection offers two methods


snow_engine.execute_statement("<SQL Statement>") # execute the SQL statement.


snow_engine.read_sql("<SQL Statement>") # execute the SQL statement and return a Pandas DataFrame.


snow_engine.write_dataframe(dataframe:pd.DataFrame, table_name:str,schema:str, index_in=False, if_exists_in="append",method_in="multi")


In [2]:
SOURCE_SCHEMA = "CUEBIQ_DATA.PAAS_CDA_PE_V3"
STOP_TABLE = f"{SOURCE_SCHEMA}.STOP_BY_EVENT_DATE_UPLEVELLED"
TMP_SCHEMA = "DEDICATED.EDDIT_EAST_BAY_TMP"
PROVIDER_ID, COUNTRY_CODE = "PAPA", "US"
MAX_RAW_STOP_MINUTES = 8*60
PILOT_WINDOWS = [(20190107,20190113),(20240108,20240114)]
PILOT_CORRIDOR_NAMES = ["Fruitvale"]
CORRIDOR_SOURCES = [{
    "path":"./corridors/OAK_BIDs", "id_col":"FID", "name_col":"BID",
    "source":"Oakland BID", "geohash_cover":"./corridors/geohash_level9_oak_bids.csv",
}]
EXPORT_DIR=Path('./activity/pilot_raw')
EXPORT_CSV=True
REUSE_EXISTING_LOOKUP=True
REBUILD_PILOT_TABLES=False
OLD_PREFIX=f"{TMP_SCHEMA}.TMP_EB_QUICK_"
T={
 'CORRIDOR_GEOHASH9':f"{TMP_SCHEMA}.TMP_EB_PILOT_CORRIDOR_GEOHASH9",
 'STOP_ROWS':OLD_PREFIX+'STOP_ROWS',
 'DEVICE_DAY_PERIOD':OLD_PREFIX+'DEVICE_DAY_PERIOD',
 'DEVICE_MONTH_PERIOD':OLD_PREFIX+'DEVICE_MONTH_PERIOD',
 'DEVICE_HALF_PERIOD':OLD_PREFIX+'DEVICE_HALF_PERIOD',
 'CORRIDOR_STOPS_RAW':OLD_PREFIX+'CORRIDOR_STOPS_RAW',
 'CORRIDOR_REPEAT_RAW':OLD_PREFIX+'CORRIDOR_REPEAT_VISITORS_RAW',
 'CORRIDOR_COVER_DIAGNOSTICS':OLD_PREFIX+'CORRIDORS_GEOHASH9_RAW_DIAGNOSTICS',
}
print('Pilot windows:',PILOT_WINDOWS,'Corridor:',PILOT_CORRIDOR_NAMES)

Pilot windows: [(20190107, 20190113), (20240108, 20240114)] Corridor: ['Fruitvale']


## 2. Helpers and input validation

In [3]:
def run(label, query, quiet=False):
    start = time.time()
    if not quiet:
        print(f'Running {label} ...')
    result = snow_engine.read_sql(query)
    if not quiet:
        print(f"{label}: {time.time()-start:.1f}s")
    return result


def existing_rows(key):
    """Return persisted row count, or None if the table does not exist."""
    try:
        return int(snow_engine.read_sql(f"SELECT COUNT(*) AS n FROM {T[key]}").iloc[0,0])
    except Exception:
        return None


def reuse_group(keys):
    counts={key:existing_rows(key) for key in keys}
    if REBUILD_PILOT_TABLES or any(n is None for n in counts.values()):
        return False
    print('Reusing completed tables:', ', '.join(f'{key}={n:,}' for key,n in counts.items()))
    return True


def sql_values(values):
    return ', '.join(f"({int(x)})" for x in values)


def sql_str_list(values):
    return '(' + ','.join("'" + str(v).replace("'", "''") + "'" for v in values) + ')'


def windows_predicate(column):
    return '(' + ' OR '.join(f"{column} BETWEEN {a} AND {b}" for a,b in PILOT_WINDOWS) + ')'


def upload(df, table, types, chunk_size=2000):
    cols = list(types)
    start = time.time()
    run(f'Create {table}', f"CREATE OR REPLACE TABLE {table} (" +
        ', '.join(f'{c} {t}' for c,t in types.items()) + ')', quiet=True)
    for offset in range(0,len(df),chunk_size):
        records = []
        for row in df.iloc[offset:offset+chunk_size][cols].itertuples(index=False, name=None):
            vals=[]
            for value in row:
                if pd.isna(value): vals.append('NULL')
                elif isinstance(value,(int,float,np.integer,np.floating)) and not isinstance(value,bool):
                    vals.append(str(value))
                else: vals.append("'" + str(value).replace("'", "''") + "'")
            records.append('(' + ','.join(vals) + ')')
        run(f'Insert rows {offset+1}–{offset+len(records)}',
            f"INSERT INTO {table} ({','.join(cols)}) VALUES " + ','.join(records), quiet=True)
    assert int(run('Verify upload', f'SELECT COUNT(*) AS n FROM {table}', quiet=True).iloc[0,0]) == len(df)
    print(f"Uploaded {len(df):,} rows to {table.split('.')[-1]} in {time.time()-start:.1f}s")


def canonical_source(config):
    path=Path(config['path'])
    if path.is_dir():
        path=path/(path.name+'.shp')
    g=gpd.read_file(path)
    required={config['id_col'],config['name_col']}
    if required-set(g): raise ValueError(f'{path}: missing {required-set(g)}')
    g=g[[config['id_col'],config['name_col'],'geometry']].copy()
    if g.crs is None or g.geometry.isna().any() or (~g.geometry.is_valid).any():
        raise ValueError(f'{path}: CRS, geometry or validity problem')
    g=g.rename(columns={config['id_col']:'local_id',config['name_col']:'corridor_name'})
    g['local_id']=g['local_id'].astype(str)
    g['corridor_name']=g['corridor_name'].astype(str).str.strip()
    g['corridor_source']=config['source']
    g['corridor_id']=config['source']+':'+g['local_id']
    cover=pd.read_csv(config['geohash_cover'],dtype={'corridor_id':str,'geohash9':str})
    expected={'corridor_id','corridor_name','geohash9','coverage_fraction'}
    if expected-set(cover): raise ValueError(f"{config['geohash_cover']}: missing {expected-set(cover)}")
    cover=cover.rename(columns={'corridor_id':'local_id'})
    cover['local_id']=cover['local_id'].astype(str)
    cover['corridor_source']=config['source']
    cover['corridor_id']=config['source']+':'+cover['local_id']
    cover['geohash9']=cover['geohash9'].str.lower().str.strip()
    cover['coverage_fraction']=pd.to_numeric(cover['coverage_fraction'])
    if cover.duplicated(['corridor_id','geohash9']).any(): raise ValueError('Duplicate corridor-geohash9')
    if not cover.geohash9.str.len().eq(9).all() or not cover.coverage_fraction.between(0,1).all():
        raise ValueError('Invalid corridor geohash or coverage fraction')
    identity=g[['corridor_id','corridor_name']].drop_duplicates()
    if identity.corridor_id.duplicated().any(): raise ValueError('Duplicate corridor ID in geometry')
    if not cover[['corridor_id','corridor_name']].drop_duplicates().merge(identity,how='left',
        on=['corridor_id','corridor_name'],indicator=True)._merge.eq('both').all():
        raise ValueError('Cover ID/name does not match geometry')
    return g,cover

sources=[canonical_source(c) for c in CORRIDOR_SOURCES]
geo=pd.concat([x[0] for x in sources],ignore_index=True)
cover=pd.concat([x[1] for x in sources],ignore_index=True)
if geo.corridor_id.duplicated().any() or cover.duplicated(['corridor_id','geohash9']).any():
    raise ValueError('IDs / corridor-geohash pairs collide across sources')
chosen=geo.loc[geo.corridor_name.isin(PILOT_CORRIDOR_NAMES),'corridor_id']
if len(chosen)!=len(PILOT_CORRIDOR_NAMES): raise ValueError('Pilot corridor names missing or ambiguous')
cover=cover.loc[cover.corridor_id.isin(chosen)].copy()
print('Corridors:',len(chosen),'pilot cover rows:',len(cover))
display(cover.groupby(['corridor_id','corridor_name']).agg(cells=('geohash9','size'),
    min_coverage=('coverage_fraction','min')).reset_index())

Corridors: 1 pilot cover rows: 20594


,corridor_id,corridor_name,cells,min_coverage
0,Oakland BID:8,Fruitvale,20594,0.800239


## 3. Use or upload the Fruitvale geohash9 lookup

The earlier lookup upload is reused when it contains the expected number of Fruitvale cells. This stage has no source-table scan.

In [4]:
run('Ensure working schema',f'CREATE SCHEMA IF NOT EXISTS {TMP_SCHEMA}')
pilot_ids=sorted(chosen.tolist())
pilot_ids_sql=sql_str_list(pilot_ids)
expected=len(cover)
try:
    present=int(run('Check corridor cover',f"SELECT COUNT(*) AS n FROM {T['CORRIDOR_GEOHASH9']} WHERE corridor_id IN {pilot_ids_sql}",quiet=True).iloc[0,0]) if REUSE_EXISTING_LOOKUP else -1
except Exception:
    present=-1
if present==expected:
    print(f'Reusing Fruitvale geohash9 lookup ({expected:,} cells)')
else:
    print(f'Uploading Fruitvale geohash9 lookup ({expected:,} cells)')
    upload(cover[['corridor_id','corridor_name','geohash9','coverage_fraction']],T['CORRIDOR_GEOHASH9'],
           {'corridor_id':'VARCHAR','corridor_name':'VARCHAR','geohash9':'VARCHAR','coverage_fraction':'FLOAT'})
run('Build cover diagnostics',f"""
CREATE OR REPLACE TABLE {T['CORRIDOR_COVER_DIAGNOSTICS']} AS
SELECT c.*,COUNT(*) OVER (PARTITION BY geohash9) AS matched_corridor_count
FROM {T['CORRIDOR_GEOHASH9']} c WHERE c.corridor_id IN {pilot_ids_sql}
""")

Running Ensure working schema ...
Ensure working schema: 1.2s
Reusing Fruitvale geohash9 lookup (20,594 cells)
Running Build cover diagnostics ...
Build cover diagnostics: 1.6s


,status
0,Table TMP_EB_QUICK_CORRIDORS_GEOHASH9_RAW_DIAG...


## 4. Deduplicate source stops and count device-days

Reuse the prior pilot tables when present. If they are missing, this cell performs the two-week source extraction and builds the monthly and half-year device summaries. A raw stop is a qualifying source observation; a visit is one device × corridor × local date. Each stop contributes to `all` and any matching time period.

In [5]:
if reuse_group(['STOP_ROWS', 'DEVICE_DAY_PERIOD', 'DEVICE_MONTH_PERIOD', 'DEVICE_HALF_PERIOD']):
    pass
else:
    run('Deduplicate pilot corridor stop rows', f"""
    CREATE OR REPLACE TABLE {T['STOP_ROWS']} AS
    WITH matched AS (
      SELECT c.corridor_id,c.corridor_name,s.cuebiq_id,s.geohash_id AS geohash9,
        TO_TIMESTAMP_TZ(s.stop_zoned_datetime) AS stop_start_ts,
        s.dwell_time_minutes,s.event_date,s.processing_date,
        ROW_NUMBER() OVER (PARTITION BY c.corridor_id,s.cuebiq_id,s.geohash_id,
          s.stop_zoned_datetime,s.dwell_time_minutes
          ORDER BY s.processing_date DESC,s.event_date ASC) AS rn
      FROM {STOP_TABLE} s JOIN {T['CORRIDOR_GEOHASH9']} c ON s.geohash_id=c.geohash9
      WHERE s.provider_id='{PROVIDER_ID}' AND s.country_code='{COUNTRY_CODE}'
        AND {windows_predicate('s.event_date')}
        AND c.corridor_id IN {pilot_ids_sql}
        AND s.stop_zoned_datetime IS NOT NULL
        AND s.dwell_time_minutes > 0 AND s.dwell_time_minutes <= {MAX_RAW_STOP_MINUTES}
    )
    SELECT corridor_id,corridor_name,cuebiq_id,geohash9,stop_start_ts,dwell_time_minutes,
      TO_NUMBER(TO_CHAR(stop_start_ts,'YYYYMMDD')) AS visit_date,
      TO_NUMBER(TO_CHAR(stop_start_ts,'YYYYMM')) AS year_month,
      CONCAT(YEAR(stop_start_ts),'H',IFF(MONTH(stop_start_ts)<=6,'1','2')) AS half_year,
      DAYOFWEEKISO(stop_start_ts) AS dow_iso,HOUR(stop_start_ts) AS hour_of_day
    FROM matched WHERE rn=1
      AND {windows_predicate("TO_NUMBER(TO_CHAR(stop_start_ts,'YYYYMMDD'))")}
    """)
    # One row per corridor/device/date/time-period, with source stops retained as a count.
    run('Build device-day visits by time period',f"""
    CREATE OR REPLACE TABLE {T['DEVICE_DAY_PERIOD']} AS
    WITH p AS (
     SELECT corridor_id,corridor_name,cuebiq_id,visit_date,year_month,half_year,'all' AS time_period FROM {T['STOP_ROWS']}
     UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,year_month,half_year,'weekdays' FROM {T['STOP_ROWS']} WHERE dow_iso BETWEEN 1 AND 5
     UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,year_month,half_year,'weekends' FROM {T['STOP_ROWS']} WHERE dow_iso IN (6,7)
     UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,year_month,half_year,'nine-five' FROM {T['STOP_ROWS']} WHERE dow_iso BETWEEN 1 AND 5 AND hour_of_day BETWEEN 9 AND 16
     UNION ALL SELECT corridor_id,corridor_name,cuebiq_id,visit_date,year_month,half_year,'evening' FROM {T['STOP_ROWS']} WHERE hour_of_day BETWEEN 17 AND 22
    )
    SELECT corridor_id,corridor_name,cuebiq_id,visit_date,year_month,half_year,time_period,
      COUNT(*) AS raw_stop_count,1 AS visit_count
    FROM p GROUP BY 1,2,3,4,5,6,7
    """)
    run('Monthly device-period records',f"""
    CREATE OR REPLACE TABLE {T['DEVICE_MONTH_PERIOD']} AS
    SELECT corridor_id,corridor_name,year_month,time_period,cuebiq_id,
      SUM(raw_stop_count) AS raw_stop_count,COUNT(*) AS visit_count
    FROM {T['DEVICE_DAY_PERIOD']} GROUP BY 1,2,3,4,5
    """)
    run('Half-year device-period records',f"""
    CREATE OR REPLACE TABLE {T['DEVICE_HALF_PERIOD']} AS
    SELECT corridor_id,corridor_name,half_year,time_period,cuebiq_id,
      SUM(raw_stop_count) AS raw_stop_count,COUNT(*) AS visit_count,
      COUNT(DISTINCT visit_date) AS distinct_visit_days,COUNT(DISTINCT year_month) AS active_months
    FROM {T['DEVICE_DAY_PERIOD']} GROUP BY 1,2,3,4,5
    """)

Reusing completed tables: STOP_ROWS=5,225, DEVICE_DAY_PERIOD=10,881, DEVICE_MONTH_PERIOD=9,661, DEVICE_HALF_PERIOD=9,661


In [6]:
# Verify that reused or rebuilt intermediates belong to these pilot windows and corridor.
counts={key:existing_rows(key) for key in ['STOP_ROWS','DEVICE_DAY_PERIOD','DEVICE_MONTH_PERIOD','DEVICE_HALF_PERIOD']}
display(pd.DataFrame([{'table':k,'rows':v} for k,v in counts.items()]))
if any(v is None or v==0 for v in counts.values()):
    raise RuntimeError('A required activity intermediate is missing or empty. Review Section 4.')
check=run('Check stop-row scope',f"""
SELECT COUNT(DISTINCT corridor_id) AS corridor_count,
 MIN(visit_date) AS first_date,MAX(visit_date) AS last_date,
 COUNT_IF(NOT {windows_predicate('visit_date')}) AS out_of_window_rows,
 COUNT_IF(corridor_id NOT IN {pilot_ids_sql}) AS wrong_corridor_rows
FROM {T['STOP_ROWS']}
""")
display(check)
if int(check.iloc[0,0])!=len(pilot_ids) or int(check.iloc[0,3]) or int(check.iloc[0,4]):
    raise RuntimeError('Cached stop rows do not match the configured pilot. Set REBUILD_PILOT_TABLES=True and rerun Section 4.')

,table,rows
0,STOP_ROWS,5225
1,DEVICE_DAY_PERIOD,10881
2,DEVICE_MONTH_PERIOD,9661
3,DEVICE_HALF_PERIOD,9661


Running Check stop-row scope ...
Check stop-row scope: 0.4s


,corridor_count,first_date,last_date,out_of_window_rows,wrong_corridor_rows
0,1,20240108,20240114,0,0


## 5. Write raw analytical outputs

`corridor_stops_raw` has one row per corridor × month × time period; `corridor_repeat_visitors_raw` has one row per corridor × half-year × time period. No normalized fields are calculated.

In [7]:
run('Write raw monthly corridor activity',f"""
CREATE OR REPLACE TABLE {T['CORRIDOR_STOPS_RAW']} AS
WITH monthly AS (
 SELECT corridor_id,corridor_name,year_month,time_period,
  SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,
  COUNT(*) AS unique_devices
 FROM {T['DEVICE_MONTH_PERIOD']} GROUP BY 1,2,3,4
)
SELECT *,raw_stop_count/NULLIF(unique_devices,0) AS raw_stops_per_unique_device,
 visit_count/NULLIF(unique_devices,0) AS visits_per_unique_device
FROM monthly
""")
run('Write raw repeat visitors',f"""
CREATE OR REPLACE TABLE {T['CORRIDOR_REPEAT_RAW']} AS
SELECT corridor_id,corridor_name,half_year,
 TO_NUMBER(SUBSTR(half_year,1,4))*10000+IFF(RIGHT(half_year,1)='1',101,701) AS period_start,
 TO_NUMBER(SUBSTR(half_year,1,4))*10000+IFF(RIGHT(half_year,1)='1',630,1231) AS period_end,
 time_period,SUM(raw_stop_count) AS raw_stop_count,SUM(visit_count) AS visit_count,
 COUNT(*) AS unique_devices,COUNT_IF(distinct_visit_days>=2) AS repeat_devices,
 COUNT_IF(distinct_visit_days>=2)/NULLIF(COUNT(*),0) AS repeat_device_prop,
 SUM(visit_count)/NULLIF(COUNT(*),0) AS visits_per_unique_device,
 COUNT_IF(distinct_visit_days>=3) AS devices_3plus_visit_days,
 COUNT_IF(active_months>=2) AS devices_2plus_active_months
FROM {T['DEVICE_HALF_PERIOD']} GROUP BY 1,2,3,6
""")

Running Write raw monthly corridor activity ...
Write raw monthly corridor activity: 0.9s
Running Write raw repeat visitors ...
Write raw repeat visitors: 1.0s


,status
0,Table TMP_EB_QUICK_CORRIDOR_REPEAT_VISITORS_RA...


## 6. QA and pilot results

In [8]:
for key,grain in [
 ('DEVICE_DAY_PERIOD','corridor_id,cuebiq_id,visit_date,time_period'),
 ('CORRIDOR_STOPS_RAW','corridor_id,year_month,time_period'),
 ('CORRIDOR_REPEAT_RAW','corridor_id,half_year,time_period')]:
    d=run('Duplicate grain: '+key,f"SELECT COUNT(*) AS n FROM (SELECT {grain} FROM {T[key]} GROUP BY {grain} HAVING COUNT(*)>1)",quiet=True)
    assert int(d.iloc[0,0])==0,key
print('Output grains have no duplicates.')
checks=run('Visit count invariants',f"""
SELECT COUNT_IF(visit_count>raw_stop_count OR unique_devices>visit_count) AS invalid_rows,
 COUNT(*) AS monthly_rows
FROM {T['CORRIDOR_STOPS_RAW']}
""")
display(checks)
assert int(checks.iloc[0,0])==0
checks=run('Daily visit invariant',f"""
SELECT COUNT_IF(visit_count<>1 OR raw_stop_count<1) AS invalid_rows,
 COUNT(*) AS device_day_period_rows
FROM {T['DEVICE_DAY_PERIOD']}
""")
display(checks)
assert int(checks.iloc[0,0])==0
display(run('Raw monthly pilot comparison',f"""
SELECT corridor_name,year_month,time_period,raw_stop_count,visit_count,
 unique_devices,visits_per_unique_device
FROM {T['CORRIDOR_STOPS_RAW']}
ORDER BY year_month,time_period
"""))
display(run('Repeat visitor pilot comparison',f"""
SELECT corridor_name,half_year,time_period,visit_count,unique_devices,
 repeat_devices,repeat_device_prop
FROM {T['CORRIDOR_REPEAT_RAW']}
ORDER BY half_year,time_period
"""))

Output grains have no duplicates.
Running Visit count invariants ...
Visit count invariants: 0.4s


,invalid_rows,monthly_rows
0,0,5


Running Daily visit invariant ...
Daily visit invariant: 0.4s


,invalid_rows,device_day_period_rows
0,0,10881


Running Raw monthly pilot comparison ...
Raw monthly pilot comparison: 0.4s


,corridor_name,year_month,time_period,raw_stop_count,visit_count,unique_devices,visits_per_unique_device
0,Fruitvale,202401,all,5225,4121,3537,1.165112
1,Fruitvale,202401,evening,1405,1254,1173,1.069054
2,Fruitvale,202401,nine-five,1718,1385,1236,1.120550
3,Fruitvale,202401,weekdays,3642,2866,2518,1.138205
4,Fruitvale,202401,weekends,1583,1255,1197,1.048454


Running Repeat visitor pilot comparison ...
Repeat visitor pilot comparison: 0.4s


,corridor_name,half_year,time_period,visit_count,unique_devices,repeat_devices,repeat_device_prop
0,Fruitvale,2024H1,all,4121,3537,416,0.117614
1,Fruitvale,2024H1,evening,1254,1173,66,0.056266
2,Fruitvale,2024H1,nine-five,1385,1236,113,0.091424
3,Fruitvale,2024H1,weekdays,2866,2518,262,0.104051
4,Fruitvale,2024H1,weekends,1255,1197,58,0.048454


## 7. Export pilot CSVs

In [9]:
if EXPORT_CSV:
    EXPORT_DIR.mkdir(parents=True,exist_ok=True)
    outputs={
      'CORRIDOR_STOPS_RAW':'corridor_stops_raw.csv',
      'CORRIDOR_REPEAT_RAW':'corridor_repeat_visitors_raw.csv',
      'CORRIDOR_COVER_DIAGNOSTICS':'corridors_geohash9_raw_diagnostics.csv',
    }
    for key,filename in outputs.items():
        df=run('Export '+key,f'SELECT * FROM {T[key]}')
        path=EXPORT_DIR/filename
        df.to_csv(path,index=False)
        print(f'{len(df):,} rows → {path}')

Running Export CORRIDOR_STOPS_RAW ...
Export CORRIDOR_STOPS_RAW: 0.4s
5 rows → activity/pilot_raw/corridor_stops_raw.csv
Running Export CORRIDOR_REPEAT_RAW ...
Export CORRIDOR_REPEAT_RAW: 0.4s
5 rows → activity/pilot_raw/corridor_repeat_visitors_raw.csv
Running Export CORRIDOR_COVER_DIAGNOSTICS ...
Export CORRIDOR_COVER_DIAGNOSTICS: 4.2s
20,594 rows → activity/pilot_raw/corridors_geohash9_raw_diagnostics.csv


## Interpretation and next step

The two weeks are a **pipeline test**, not a representative 2019-to-2024 trend. Unique devices are deduplicated within a corridor/month/time period. Repeat visitor IDs are interpreted only within half-years, but the observed pilot weeks do not cover complete half-years. A stop can belong to several time periods; do not add them together. Corridor edge cells from the prefiltered geohash9 cover are counted equally.

The production activity run can extend the same stop and device-day transformations to all corridors and months of 2019–2025, ideally in bounded month or quarter batches. HOME origins and sample-based normalization need a source with usable US HOME geographies and historical coverage; add those as a separate stage when resolved. The raw CSVs should never be described as panel-normalized.